# CausalMan: RCA data generation

This notebook generates the interventional datasets for the CausalMan root-cause analysis (RCA) benchmark.
The RCA task asks: given samples drawn from a system under an unknown intervention, identify which
variable(s) were intervened upon and characterise the intervention.

There are four tasks, each run on two scales, giving eight datasets in total:

| Task | Scales | Scenario |
|---|---|---|
| s01 | micro, small | Hard (atomic) intervention: press-fitting force locked to 17 000 N |
| s02 | micro, small | Soft intervention: max allowable force drawn from Normal(18 500, 3 000) |
| s03 | medium, large | Soft intervention on both T1 and T2 max force simultaneously |
| s04 | medium, large | Mixed: two observed soft interventions + one hidden hard intervention on `MV1_Emv` |

Each dataset folder contains:

| File | Description |
|---|---|
| `causalman_<scale>_do(...).csv` | Observable node samples under the named intervention |
| `intervention_mask.csv` | Boolean per-row mask — `True` where the intervention was applied |
| `interventions.json` | Machine-readable intervention spec (variable, kind, parameters) |

**Edit the configuration cell below, then Run All.**

In [ ]:
# ── The only cell you need to edit ────────────────────────────────────────────

# Choose which tasks and scales to generate.
# Tasks are automatically skipped for scales outside their compatible set
# (s01/s02 → micro/small; s03/s04 → medium/large).
TASK_IDS    = ["s02"]        # any subset of ["s01", "s02", "s03", "s04"]
SCALES      = ["micro", "small"]      # any subset of ["micro", "small", "medium", "large"]
SEED        = 42
N_SAMPLES   = 10_000
OUTPUT_ROOT = "output/causalman_rca"

# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
import json
import os
from pathlib import Path
import sys

from sympy.stats import Normal

# Put the repository root before this notebook directory so causalman.py
# cannot shadow the causalman package when the notebook runs in-place.
PROJECT_ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
                     if (path / "pyproject.toml").is_file()
                     and (path / "causalman" / "__init__.py").is_file()), None)
if PROJECT_ROOT is not None:
    project_root = str(PROJECT_ROOT)
    if project_root in sys.path:
        sys.path.remove(project_root)
    sys.path.insert(0, project_root)

from causalman import CausalMan

# ── Task registry ─────────────────────────────────────────────────────────────
# Each entry defines one RCA scenario. "scales" lists the CausalMan variants
# compatible with that task; tasks are skipped for scales not in SCALES.
#
# kind="constant" → hard/atomic intervention  do(X = value)
# kind="normal"   → soft/stochastic intervention  do(X ~ Normal(mean, std))
TASKS = [
    {
        # s01: T1 press-fitting force fixed to an abnormally high constant.
        # Hard intervention — the variable is fully determined, no noise.
        "task_id": "s01",
        "slug":    "force_17000",
        "scales":  ["micro", "small"],
        "interventions": [
            {"variable": "PF_M1_T1_Force", "kind": "constant", "value": 17000.0},
        ],
    },
    {
        # s02: Max allowable press-fitting force shifted to a higher distribution.
        # Soft intervention — the variable retains its stochasticity.
        "task_id": "s02",
        "slug":    "fmax_normal",
        "scales":  ["micro", "small"],
        "interventions": [
            {"variable": "PF_M1_T1_Fmax", "kind": "normal", "mean": 18500.0, "std": 3000.0},
        ],
    },
    {
        # s03: Both T1 and T2 max forces shifted simultaneously — multi-target soft intervention.
        "task_id": "s03",
        "slug":    "two_fmax_normal",
        "scales":  ["medium", "large"],
        "interventions": [
            {"variable": "PF_M1_T1_Fmax", "kind": "normal", "mean": 18500.0, "std": 3000.0},
            {"variable": "PF_M1_T2_Fmax", "kind": "normal", "mean": 19500.0, "std": 4000.0},
        ],
    },
    {
        # s04: Mixed scenario — two observed soft interventions plus one hidden hard
        # intervention on MV1_Emv (a latent node absent from the observed CSV).
        # Tests whether methods can detect a root cause they cannot directly observe.
        "task_id": "s04",
        "slug":    "mixed_with_hidden_emv",
        "scales":  ["medium", "large"],
        "interventions": [
            {"variable": "MV2_DmvMax",     "kind": "normal",   "mean": 4.7,     "std": 1.0},
            {"variable": "PF_M1_T1_Force", "kind": "normal",   "mean": 16500.0, "std": 3000.0},
            # MV1_Emv is latent — it does not appear in the observed CSV.
            {"variable": "MV1_Emv",        "kind": "constant", "value": 190000.0},
        ],
    },
]

In [ ]:
from datetime import datetime

now = datetime.now().strftime("%Y_%m_%d_%H%M%S")
OUTPUT_ROOT = f"{OUTPUT_ROOT}_{now}"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

for task in [task for task in TASKS if task["task_id"] in TASK_IDS]:
    for scale in [scale for scale in task["scales"] if scale in SCALES]:
        dataset_id = f"rca_{task['task_id']}_{scale}_{task['slug']}"
        out_dir = os.path.join(OUTPUT_ROOT, dataset_id)
        os.makedirs(out_dir, exist_ok=True)
        print(f"\n── {dataset_id} ──")

        intervention_dict = {}
        for intervention in task["interventions"]:
            if intervention["kind"] == "constant":
                intervention_dict[intervention["variable"]] = intervention["value"]
            else:
                intervention_dict[intervention["variable"]] = Normal(
                    intervention["variable"],
                    intervention["mean"],
                    intervention["std"],
                )

        simulator = CausalMan(
            name=f"causalman_{scale}",
            seed=SEED,
            parallelize=True,
            save_path=os.path.join(out_dir, "_simulator"),
        )
        simulator.intervention_dict = intervention_dict
        dataset, _, _, _, _, intervention_mask = simulator.sample(
            n_samples=N_SAMPLES
        )

        do_parts = []
        for intervention in task["interventions"]:
            if intervention["kind"] == "constant":
                do_parts.append(
                    f"{intervention['variable']}={intervention['value']}"
                )
            else:
                do_parts.append(
                    f"{intervention['variable']}="
                    f"Normal({intervention['mean']},{intervention['std']})"
                )
        do_str = "do(" + ",".join(do_parts) + ")"

        dataset.to_csv(
            os.path.join(out_dir, f"causalman_{scale}_{do_str}.csv"),
            index=False,
        )
        intervention_mask.to_csv(
            os.path.join(out_dir, "intervention_mask.csv"), index=False
        )

        with open(os.path.join(out_dir, "interventions.json"), "w") as file:
            json.dump(task["interventions"], file, indent=2)

        targets = [
            intervention["variable"] for intervention in task["interventions"]
        ]
        print(
            f"  {len(dataset):,} rows | "
            f"{dataset.shape[1]} observable columns"
        )
        print(f"  intervention targets: {targets}")

print(f"\nDone → {os.path.abspath(OUTPUT_ROOT)}")